# Phase 7: YOLOv12 Optimization Study (Part 2 - Threshold Sweeps)

This notebook evaluates the best trained weights from EXPs 5-8 across various Confidence and NMS IoU thresholds using ONLY the validation set.

In [8]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
# Install dependencies
!pip install ultralytics

In [10]:
import os
import sys
import pandas as pd
from ultralytics import YOLO

PROJECT_ROOT = '/content/drive/MyDrive/sem_defect_project'
os.chdir(PROJECT_ROOT)
print(f"Current working directory: {os.getcwd()}")

Current working directory: /content/drive/MyDrive/sem_defect_project


In [13]:
# IMPORTANT: Point this to the best weights from your EXP-10 run.
BEST_WEIGHTS_PATH = 'runs/detect/runs/detect/EXP-10-YOLOv12m-640-200/weights/best.pt'
DATA_YAML = 'dataset_yolo_single_class/data.yaml'

print(f"Loading weights from: {BEST_WEIGHTS_PATH}")
model = YOLO(BEST_WEIGHTS_PATH)

# Check architecture
model_yaml = getattr(model.model, 'yaml', {})
arch_name = model_yaml.get('yaml_file', 'Unknown Architecture')
print(f"\nLoaded Model Architecture: {arch_name}")



Loading weights from: runs/detect/runs/detect/EXP-10-YOLOv12m-640-200/weights/best.pt

Loaded Model Architecture: yolo11m.yaml


In [14]:
# Define sweep parameters
conf_thresholds = [0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40, 0.50, 0.60]
fixed_iou = 0.50 # Standard NMS IoU setting

results_list = []

print(f"Starting Threshold Sweeps on Validation Set (Fixed NMS IoU={fixed_iou})...")

for conf in conf_thresholds:
    print(f"\n--- Evaluating Conf: {conf} ---")
    
    # Run validation
    metrics = model.val(
        data=DATA_YAML,
        split='val',
        conf=conf,
        iou=fixed_iou,
        verbose=False
    )
    
    # Extract metrics
    precision = metrics.results_dict['metrics/precision(B)']
    recall = metrics.results_dict['metrics/recall(B)']
    map50 = metrics.results_dict['metrics/mAP50(B)']
    map50_95 = metrics.results_dict['metrics/mAP50-95(B)']
    
    # Calculate F1
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    results_list.append({
        'conf': conf,
        'iou': fixed_iou,
        'precision': precision,
        'recall': recall,
        'map50': map50,
        'map50_95': map50_95,
        'f1': f1
    })

# Create a DataFrame and save to CSV
df = pd.DataFrame(results_list)
os.makedirs('runs/detect/EXP-10-YOLOv12m-640-200', exist_ok=True)
csv_path = 'runs/detect/EXP-10-YOLOv12m-640-200/EXP-10-Threshold-Sweep-Results.csv'
df.to_csv(csv_path, index=False)

print(f"\nSweep Complete! Results saved to {csv_path}")


Starting Threshold Sweeps on Validation Set (Fixed NMS IoU=0.5)...

--- Evaluating Conf: 0.1 ---
Ultralytics 8.4.150 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLO11m summary (fused): 125 layers, 20,030,803 parameters, 0 gradients, 67.8 GFLOPs
val: Fast image access ✅ (ping: 0.5±0.1 ms, read: 63.2±18.1 MB/s, size: 58.1 KB)
val: Scanning /content/drive/MyDrive/sem_defect_project/dataset_yolo_single_class/valid/labels.cache... 31 images, 3 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 31/31 6.8Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 1.3it/s 1.5s2.5s
                   all         31        184      0.661      0.766      0.647      0.379
Speed: 4.5ms preprocess, 26.5ms inference, 0.0ms loss, 0.9ms postprocess per image
Results saved to /content/drive/MyDrive/sem_defect_project/runs/detect/val

--- Evaluating Conf: 0.15 ---
Ultralytics 8.4.150 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tes

In [16]:
# Display the top 5 configurations sorted by F1-Score
display(df.sort_values(by='f1', ascending=False).head(5))

# Also display the top 5 configurations sorted by Recall (while maintaining precision > 0.50)
print("\nTop configs for Recall (Precision > 0.50):")
filtered_df = df[df['precision'] > 0.50]
display(filtered_df.sort_values(by='recall', ascending=False).head(5))


,conf,iou,precision,recall,map50,map50_95,f1
0,0.10,0.5,0.661061,0.766304,0.647175,0.378558,0.709803
1,0.15,0.5,0.661061,0.766304,0.641215,0.374794,0.709803
2,0.20,0.5,0.658879,0.766304,0.621954,0.367185,0.708543
3,0.25,0.5,0.693122,0.711957,0.587806,0.348675,0.702413
4,0.30,0.5,0.715909,0.684783,0.566656,0.336698,0.700000



Top configs for Recall (Precision > 0.50):


,conf,iou,precision,recall,map50,map50_95,f1
0,0.10,0.5,0.661061,0.766304,0.647175,0.378558,0.709803
1,0.15,0.5,0.661061,0.766304,0.641215,0.374794,0.709803
2,0.20,0.5,0.658879,0.766304,0.621954,0.367185,0.708543
3,0.25,0.5,0.693122,0.711957,0.587806,0.348675,0.702413
4,0.30,0.5,0.715909,0.684783,0.566656,0.336698,0.700000
